<a href="https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [7]:
DIM = f"read_parquet('{REL}/dim_content.parquet')"
feature_vector = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS gsc_impressions,
        SUM(f.gsc_clicks) AS gsc_clicks,

        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,

        d.word_count,

        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days

    FROM {FEB} f

    JOIN {DIM} d
      ON f.client_hash_id = d.client_hash_id
     AND f.content_hash_id = d.content_hash_id

    WHERE f.gsc_data_available IS TRUE
      AND d.content_created_date <= DATE '2026-02-28'

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.word_count,
        d.content_created_date
""").df()

print("Feature rows:", len(feature_vector))
feature_vector.head()

Feature rows: 153559


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days
0,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.273467,2689,226
1,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,8.032653,3108,226
2,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,10.307216,2396,226
3,client_e547b89c05043229,content_cb6bc37251e57efa,688.0,0.0,13.879360,2949,226
4,client_e547b89c05043229,content_cbe43d4b6ce2d320,291.0,0.0,3.972509,2781,226


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


### Feature notes

- `gsc_impressions`: February Google Search Console impressions. Missing/unavailable GSC rows are excluded using `gsc_data_available IS TRUE`. **Available when:** after February 2026 closes.
- `gsc_clicks`: February Google Search Console clicks. Missing/unavailable GSC rows are excluded. **Available when:** after February 2026 closes.
- `gsc_avg_position`: Impression-weighted average search position during February. **Available when:** after February 2026 closes.
- `word_count`: Number of words in the content from `dim_content`. **Available when:** before the February decision cutoff.
- `content_age_days`: Number of days since content creation at the February 28, 2026 cutoff. **Available when:** at the February decision cutoff.

All five features are available before the March outcome window and therefore do not use future March information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Build March future outcome
mar = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS march_ctr
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

# Join February features with the future March outcome
dataset = feature_vector.merge(
    mar,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
).dropna()

print("Rows with February features + March outcome:", len(dataset))

# -----------------------------
# DELIBERATE LEAKAGE EXPERIMENT
# -----------------------------

# This feature is created directly from the future label.
dataset["LEAKED_MARCH_CTR"] = dataset["march_ctr"]

X = dataset[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_age_days",
    "LEAKED_MARCH_CTR"
]]

y = dataset["march_ctr"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("LEAKAGE TEST R²:", round(r2_score(y_test, pred), 4))

Rows with February features + March outcome: 58279
LEAKAGE TEST R²: 0.9999


### Leakage experiment

I deliberately added `LEAKED_MARCH_CTR`, which is exactly the future March CTR used as the label.

The resulting test R² was **0.9999**, which is an intentionally unrealistic result. This demonstrates that directly including the future label as a feature creates severe target leakage.

`LEAKED_MARCH_CTR` must therefore be removed before any honest modeling.

In [9]:
# Remove the deliberately leaked feature.
X_honest = dataset[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_age_days"
]]

y = dataset["march_ctr"]

print("Honest features:")
print(X_honest.columns.tolist())

Honest features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'word_count', 'content_age_days']


## 4. What I excluded and why

### What I excluded and why

- `LEAKED_MARCH_CTR` — deliberately created from the March outcome to demonstrate data leakage; removed before honest modeling.
- `trend_direction` — derived from outcome/trend information and could reveal the target.
- `trend_pct` — used to derive `trend_direction`, so it can leak the target.
- `is_declining_label` — directly derived from the trend/label mechanism, so it is not an independent feature.
- March/future performance fields — unavailable at the February decision cutoff and would leak future information.
- `client_hash_id` and `content_hash_id` — used for grouping and joining, not predictive features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.